# 4.3 — Robustness to Missing Stations

Masked vs visible station performance at MR=0.5 with the actual fixed
evaluation mask from the v27 dump. Controlled penalty analysis and
operational comparison with v31.

In [ ]:
import os, sys, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
for _cand in (os.getcwd(),
              os.path.join(os.getcwd(), "notebooks", "analysis"),
              os.path.dirname(os.path.abspath("__file__"))):
    if os.path.isfile(os.path.join(_cand, "common.py")):
        if _cand not in sys.path: sys.path.insert(0, _cand)
        break
import importlib, common as C; importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

RUNS  = C.discovered_runs()
MR0_RUNS = [r for r in RUNS if "mr0.00" in RUNS[r]]
ns = C.norm_stats(); VARS = ns["var_names"]; STD = ns["std"]
stn = C.station_table(); KEEP = C.keep_mask(stn, VARS)
AGG  = {r: C.load_agg(r, "mr0.00") for r in MR0_RUNS}
GRID = AGG[MR0_RUNS[0]]["grid"]; LEAD = C.lead_labels(GRID); K = len(GRID)
NV = len(VARS)
print("Models at MR=0.00:", MR0_RUNS)

# Also load MR=0.5 aggregations
MR5_RUNS = [r for r in RUNS if "mr0.50" in RUNS[r]]
AGG5 = {r: C.load_agg(r, "mr0.50") for r in MR5_RUNS}
print("Models at MR=0.50:", MR5_RUNS)
REF_KI = 6  # ≈ 3 h


In [ ]:
import geopandas as gpd
import rioxarray  # noqa

PROJ = os.path.abspath(os.path.join(os.getcwd(), "..", "..")) \
       if os.path.isfile("common.py") else os.getcwd()
if os.path.join(PROJ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PROJ, "src"))

PATH_SWISSSHAPE = os.path.expanduser(
    os.environ.get("SWISSSHAPE",
        os.path.join(PROJ, "swissboundaries3d_2056.shp.zip")))
_CAND = [os.environ.get("DATA_ROOT", ""),
         os.path.expanduser("~/PeakWeatherDataset"),
         os.path.join(PROJ, "PeakWeatherDataset")]
DATA_ROOT = next((p for p in _CAND if os.path.isdir(str(p))), _CAND[-1])

from peakweather.dataset import PeakWeatherDataset
ds_topo = PeakWeatherDataset(
    root=DATA_ROOT,
    parameters=["temperature", "pressure", "humidity",
                 "wind_speed", "wind_direction", "precipitation"],
    compute_uv=True, station_type="meteo_station",
    imputation_method=None, freq="d", extended_topo_vars="DEM")

def _load_dem_and_border(ds_topo, path_swissshape, coarsen=10):
    switzerland = gpd.read_file(
        path_swissshape,
        layer='swissBOUNDARIES3D_1_5_TLM_LANDESGEBIET').to_crs('EPSG:2056')
    minx, miny, maxx, maxy = switzerland.total_bounds
    topo = ds_topo.load_topography()
    dem  = topo['topo_DEM'].dem
    dem_ch = dem.rio.clip(switzerland.geometry, switzerland.crs, drop=False)
    dem_bg = dem.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_fg = dem_ch.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_bg = dem_bg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    dem_fg = dem_fg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    return dem_bg, dem_fg, switzerland

def draw_dem(ax, dem_bg, dem_fg, switzerland):
    norm = mcolors.Normalize(vmin=0, vmax=4500)
    dem_bg.plot(ax=ax, cmap='terrain', norm=norm, alpha=0.35,
                robust=True, add_labels=False, add_colorbar=False)
    dem_fg.plot(ax=ax, cmap='terrain', norm=norm,
                robust=True, add_labels=False, add_colorbar=False)
    switzerland.boundary.plot(ax=ax, color='white', linewidth=1.0)
    ax.axis('off')

print('Loading DEM + border ...')
dem_bg, dem_fg, switzerland = _load_dem_and_border(ds_topo, PATH_SWISSSHAPE)
print('Done.')

In [ ]:
# ── Marker convention across all MR=0.5 maps ──
# masked stations  = triangle marker
# visible stations = 'o' marker
MRK_MASKED, MRK_VISIBLE = "^", "o"
COL_MASKED, COL_VISIBLE = "#C4502A", "#1F5F6B"

## Fixed evaluation mask — station map

Which stations the encoder sees vs. which it must reconstruct,
read directly from the v27 MR=0.5 dump's `masked_idx` tensor.

In [ ]:
import torch

# Read the actual masked_idx from the v27 MR=0.5 dump
d5 = C.load_dump("v27", "mr0.50")
MI = d5["masked_idx"]                      # (M, N_masked)
N = len(stn)

# With a fixed eval mask, masked_idx is identical across all windows.
unique_masks = torch.unique(MI, dim=0)
if unique_masks.shape[0] == 1:
    print(f"Fixed evaluation mask confirmed: {MI.shape[1]} stations "
          f"masked in every window ({MI.shape[0]} windows)")
else:
    print(f"WARNING: {unique_masks.shape[0]} distinct masks found across "
          f"{MI.shape[0]} windows — mask is NOT fixed.")

masked_set = set(MI[0].tolist())
is_masked = np.array([i in masked_set for i in range(N)])
n_masked = is_masked.sum()
n_visible = N - n_masked

# ── Map ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7.5))
draw_dem(ax, dem_bg, dem_fg, switzerland)

sel_v = ~is_masked
ax.scatter(stn.easting[sel_v], stn.northing[sel_v],
           c=COL_VISIBLE, marker=MRK_VISIBLE, s=60,
           edgecolors="white", linewidths=0.5,
           label=f"visible ({n_visible})", zorder=5)
sel_m = is_masked
ax.scatter(stn.easting[sel_m], stn.northing[sel_m],
           c=COL_MASKED, marker=MRK_MASKED, s=80,
           edgecolors="white", linewidths=0.5,
           label=f"masked  ({n_masked})", zorder=6)

ax.legend(fontsize=10, framealpha=0.85, loc="upper left")
ax.set_title(f"Fixed evaluation mask — "
             f"{n_masked}/{N} stations masked (MR=0.5)", fontsize=12)
plt.tight_layout()
C.save_fig(fig, "43_fixed_eval_mask_map"); plt.show()

# ── Elevation distribution ───────────────────────────────────────────
h = stn.height.values
bins = [0, 500, 1000, 1500, 2000, 3000, 5000]
print("\nElevation distribution:")
for lo, hi in zip(bins[:-1], bins[1:]):
    band = (h >= lo) & (h < hi)
    nm = (is_masked & band).sum()
    nt = band.sum()
    print(f"  {lo:>4}–{hi:<4} m:  {nm:>3} / {nt:>3} masked  ({100*nm/max(nt,1):5.1f}%)")

print(f"\nMasked:  {', '.join(stn.abbr[is_masked])}")
print(f"Visible: {', '.join(stn.abbr[~is_masked])}")
del d5

In [ ]:
# ── Map: per-station masked MAE at +0 h, by variable ─────────────────────────
# Uses is_masked from the cell above.
a5 = AGG5["v27"]
KI_0H = 0  # lead index for +0 h

fig, axes = plt.subplots(1, NV, figsize=(5.5 * NV, 7))
for vi, v in enumerate(VARS):
    ax = axes[vi]
    draw_dem(ax, dem_bg, dem_fg, switzerland)

    # Per-station masked MAE at +0 h
    cnt_msk = a5["mod_msk_cnt"][KI_0H, :, vi]
    mae_msk = np.full(cnt_msk.shape, np.nan)
    ok = cnt_msk > 0
    mae_msk[ok] = a5["mod_msk_sum_phys"][KI_0H, ok, vi] / cnt_msk[ok]
    has_msk = is_masked & np.isfinite(mae_msk)

    # Visible stations: white circles (background)
    sel_v = ~is_masked
    ax.scatter(stn.easting[sel_v], stn.northing[sel_v],
               c="white", marker=MRK_VISIBLE, s=60,
               edgecolors="0.5", linewidths=0.5, zorder=4,
               label=f"visible ({sel_v.sum()})")

    # Masked stations: triangles coloured by MAE
    vals = mae_msk[has_msk]
    vmin, vmax = np.nanpercentile(vals, 2), np.nanpercentile(vals, 98)
    sc = ax.scatter(stn.easting[has_msk], stn.northing[has_msk],
                    c=vals, cmap="magma", vmin=vmin, vmax=vmax,
                    marker=MRK_MASKED, s=80,
                    edgecolors="0.3", linewidths=0.4, zorder=5,
                    label=f"masked ({has_msk.sum()})")
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label="MAE")

# Sync wind colour scales
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
sc_u = axes[wu_i].collections[-1]; sc_v = axes[wv_i].collections[-1]
vmin = min(sc_u.get_clim()[0], sc_v.get_clim()[0])
vmax = max(sc_u.get_clim()[1], sc_v.get_clim()[1])
sc_u.set_clim(vmin, vmax); sc_v.set_clim(vmin, vmax)

axes[0].legend(fontsize=8, loc="upper left", framealpha=0.85)
#fig.suptitle("Per-station masked MAE at +0 h (MAE Transformer, MR=0.5)\n"
#             "△ = masked (coloured by MAE), ○ = visible (white)",
#             y=1.04, fontsize=11)
fig.suptitle("MAE Transformer at MR=0.5, Δ=0 (reconstruction): per-station MAE over all windows in which the station was masked\n"
             "triangles = the 77 stations hidden in the first test window (coloured by that MAE), circles = the 78 stations visible in it", fontsize=13, y=1.02)
plt.tight_layout()
C.save_fig(fig, "43_masked_mae_map_0h"); plt.show(); plt.close(fig)

In [ ]:
# ── Scatter: per-station MAE vs height by TOD — v31 MR=0.0 & v27 MR=0.5 ────
# Only visible stations (under v27 MR=0.5 mask) are plotted.
# Open markers = v31 MR=0.0 (dense baseline), filled = v27 MR=0.5.
# One row per TOD bin (00–06, 06–12, 12–18, 18–24 UTC).
EXT0 = C.load_ext("v31", "mr0.00")   # dense baseline
EXT5 = C.load_ext("v27", "mr0.50")
h = stn.height.values

TOD_LABELS = C.TOD_LABELS  # ["00–06 UTC", "06–12 UTC", "12–18 UTC", "18–24 UTC"]
n_tod = len(TOD_LABELS)

fig, axes = plt.subplots(n_tod, NV, figsize=(4.5 * NV, 4.5 * n_tod),
                         sharey="col", sharex=True)

for ti, tod_lbl in enumerate(TOD_LABELS):
    for vi, v in enumerate(VARS):
        ax = axes[ti, vi]

        # MR=0.0: per-station MAE (tod_mod_sum / tod_mod_cnt)
        cnt0 = EXT0["tod_mod_cnt"][ti, REF_KI, :, vi]
        mae0 = np.full(cnt0.shape, np.nan)
        ok0 = cnt0 > 0
        mae0[ok0] = EXT0["tod_mod_sum"][ti, REF_KI, ok0, vi] / cnt0[ok0]

        # MR=0.5 visible
        cnt5v = EXT5["tod_mod_vis_cnt"][ti, REF_KI, :, vi]
        mae5v = np.full(cnt5v.shape, np.nan)
        ok5v = cnt5v > 0
        mae5v[ok5v] = EXT5["tod_mod_vis_sum"][ti, REF_KI, ok5v, vi] / cnt5v[ok5v]

        # MR=0.5 masked
        cnt5m = EXT5["tod_mod_msk_cnt"][ti, REF_KI, :, vi]
        mae5m = np.full(cnt5m.shape, np.nan)
        ok5m = cnt5m > 0
        mae5m[ok5m] = EXT5["tod_mod_msk_sum"][ti, REF_KI, ok5m, vi] / cnt5m[ok5m]

        vis = ~is_masked
        msk = is_masked

        # MR=0.0 — open markers
        ax.scatter(h[vis & ok0], mae0[vis & ok0],
                   marker="o", s=25, alpha=0.5, facecolors="none",
                   edgecolors=COL_VISIBLE, linewidths=0.8, zorder=4,
                   label="Dense MR=0 vis" if vi == 0 and ti == 0 else None)
        ax.scatter(h[msk & ok0], mae0[msk & ok0],
                   marker="^", s=30, alpha=0.5, facecolors="none",
                   edgecolors=COL_MASKED, linewidths=0.8, zorder=4,
                   label="Dense MR=0 msk" if vi == 0 and ti == 0 else None)

        # MR=0.5 — filled markers
        ax.scatter(h[vis & ok5v], mae5v[vis & ok5v],
                   marker="o", s=25, alpha=0.65,
                   c=COL_VISIBLE, edgecolors="white", linewidths=0.3, zorder=5,
                   label="MAE Tr. MR=0.5 vis" if vi == 0 and ti == 0 else None)
        ax.scatter(h[msk & ok5m], mae5m[msk & ok5m],
                   marker="^", s=30, alpha=0.65,
                   c=COL_MASKED, edgecolors="white", linewidths=0.3, zorder=5,
                   label="MAE Tr. MR=0.5 msk" if vi == 0 and ti == 0 else None)

        ax.grid(alpha=0.3)
        if ti == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if vi == 0:
            ax.set_ylabel(f"{tod_lbl}\nMAE", fontsize=9)
        if ti == n_tod - 1:
            ax.set_xlabel("Station height [m]", fontsize=9)

axes[0, 0].legend(fontsize=7, loc="upper left", framealpha=0.85)
fig.suptitle(f"Per-station MAE vs elevation at {LEAD[REF_KI]} — Dense vs MAE Transformer MR=0.5, by TOD\n"
             f"Open = v31 MR=0.0, filled = v27 MR=0.5  |  ○ = visible, △ = masked",
             y=1.02, fontsize=11)
plt.tight_layout()
C.save_fig(fig, "43_mae_vs_height_by_tod_v31"); plt.show(); plt.close(fig)
del EXT0, EXT5

In [ ]:
# ── Map: ΔMAE on visible stations at +3 h — MAE Transformer MR=0.5 vs Dense ────────
# ΔMAE = MAE(v27 MR=0.5 vis) − MAE(v31 MR=0.0) on same visible stations.
# Positive (red) = v27 MR=0.5 worse; negative (blue) = v27 MR=0.5 better.
# Masked stations shown as grey triangles for context.
a0 = AGG["v31"]; a5 = AGG5["v27"]

fig, axes = plt.subplots(1, NV, figsize=(5.5 * NV, 7))
fig.set_facecolor("white")

sc_by_var = {}  # store scatter handles for wind sync
for vi, v in enumerate(VARS):
    ax = axes[vi]
    draw_dem(ax, dem_bg, dem_fg, switzerland)

    # MR=0.0 per-station MAE
    cnt0 = a0["mod_all_cnt"][REF_KI, :, vi]
    mae0 = np.full(cnt0.shape, np.nan)
    ok0 = cnt0 > 0
    mae0[ok0] = a0["mod_all_sum_phys"][REF_KI, ok0, vi] / cnt0[ok0]

    # MR=0.5 visible per-station MAE
    cnt5v = a5["mod_vis_cnt"][REF_KI, :, vi]
    mae5v = np.full(cnt5v.shape, np.nan)
    ok5 = cnt5v > 0
    mae5v[ok5] = a5["mod_vis_sum_phys"][REF_KI, ok5, vi] / cnt5v[ok5]

    # Same-station comparison
    valid = ~is_masked & ok0 & ok5
    diff = mae5v[valid] - mae0[valid]

    vabs = np.nanpercentile(np.abs(diff), 98)
    sc = ax.scatter(stn.easting[valid], stn.northing[valid],
                    c=diff, cmap="RdBu_r", vmin=-vabs, vmax=vabs,
                    marker="o", s=60,
                    edgecolors="0.3", linewidths=0.4, zorder=5)
    sc_by_var[vi] = sc
    # Masked stations: grey triangles
    ax.scatter(stn.easting[is_masked], stn.northing[is_masked],
               c="0.75", marker="^", s=80,
               edgecolors="0.5", linewidths=0.3, zorder=4)
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label="ΔMAE")

# Sync wind colour scales using stored scatter handles
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
vabs = max(abs(sc_by_var[wu_i].get_clim()[0]), abs(sc_by_var[wu_i].get_clim()[1]),
           abs(sc_by_var[wv_i].get_clim()[0]), abs(sc_by_var[wv_i].get_clim()[1]))
sc_by_var[wu_i].set_clim(-vabs, vabs)
sc_by_var[wv_i].set_clim(-vabs, vabs)

REF_LEAD = LEAD[REF_KI]
#fig.suptitle(f"ΔMAE on visible stations at {REF_LEAD} — MAE Transformer MR=0.5 vs Dense\n"
#             f"ΔMAE = MAE(v27 MR=0.5) − MAE(v31 MR=0.0)  |  "
#             f"red = v27 worse, blue = v27 better\n"
#             f"△ grey = masked stations",
#             y=1.04, fontsize=11)
fig.suptitle(f"ΔMAE at {REF_LEAD}: MAE Transformer (MR=0.5, visible windows) − Dense (MR=0), 78 stations visible in the first test window\n"
             "red = masked-trained model worse; grey triangles = stations masked in that window", fontsize=13, y=1.02)
plt.tight_layout()
C.save_fig(fig, "43_vis_dmae_map_v27mr5_minus_v31mr0"); plt.show(); plt.close(fig)

In [ ]:
# ── Summary: visible-station MAE — v31 MR=0.0 (dense) vs v27 MR=0.5 at +3 h ─
# Apple-to-apple: only stations that are VISIBLE under v27 MR=0.5 and have
# valid counts in BOTH v31 MR=0.0 and v27 MR=0.5.
a0 = AGG["v31"]; a5 = AGG5["v27"]

rows = []
for vi, v in enumerate(VARS):
    # v31 MR=0.0 per-station (dense baseline, all stations visible)
    cnt0 = a0["mod_all_cnt"][REF_KI, :, vi]
    sum0 = a0["mod_all_sum_phys"][REF_KI, :, vi]

    # v27 MR=0.5 visible-station subset
    cnt5 = a5["mod_vis_cnt"][REF_KI, :, vi]
    sum5 = a5["mod_vis_sum_phys"][REF_KI, :, vi]

    # Intersection: visible in MR=0.5 AND valid data in both evals
    valid = ~is_masked & (cnt0 > 0) & (cnt5 > 0)

    mae0 = sum0[valid] / cnt0[valid]
    mae5 = sum5[valid] / cnt5[valid]
    d = mae5 - mae0  # positive = MR=0.5 worse

    # Pooled MAE over the SAME station set
    pooled_0 = sum0[valid].sum() / cnt0[valid].sum()
    pooled_5 = sum5[valid].sum() / cnt5[valid].sum()

    rows.append({
        "Variable": v,
        "Unit": C.UNITS[v],
        "N stations": int(valid.sum()),
        "Pooled MAE (v31 MR=0)": pooled_0,
        "Pooled MAE (v27 MR=0.5)": pooled_5,
        "ΔMAE (pooled)": pooled_5 - pooled_0,
        "Δ% (pooled)": 100 * (pooled_5 - pooled_0) / pooled_0,
        "Mean ΔMAE (per-stn)": np.mean(d),
        "Median ΔMAE": np.median(d),
        "Std ΔMAE": np.std(d),
        "Stations improved": int((d < 0).sum()),
        "Stations worsened": int((d > 0).sum()),
    })

df_summary = pd.DataFrame(rows)
REF_LEAD = LEAD[REF_KI]
print(f"Visible-station MAE: v31 MR=0.0 vs v27 MR=0.5 at {REF_LEAD}")
print(f"Compared on the same {rows[0]['N stations']} visible stations\n")
display(df_summary.style.format({
    "Pooled MAE (v31 MR=0)": "{:.4f}",
    "Pooled MAE (v27 MR=0.5)": "{:.4f}",
    "ΔMAE (pooled)": "{:+.4f}",
    "Δ% (pooled)": "{:+.2f}%",
    "Mean ΔMAE (per-stn)": "{:+.4f}",
    "Median ΔMAE": "{:+.4f}",
    "Std ΔMAE": "{:.4f}",
}).set_caption(f"ΔMAE = MAE(v27 MR=0.5 vis) − MAE(v31 MR=0.0) on identical visible station set  —  "
               f"positive = degradation"))
C.save_table(df_summary, "43_vis_mae_change_v31mr0_vs_v27mr5")

In [ ]:
# ── Nearest-neighbour distances for visible stations under MR=0.0 vs MR=0.5 ──
# MR=0.0: all 155 stations visible → NN distances from full network.
# MR=0.5: masked stations removed → NN distances from visible subset only.
east = stn.easting.values; north = stn.northing.values
N = len(stn)

# Pairwise distance matrix (km)
DIST = np.sqrt((east[:, None] - east[None, :]) ** 2 +
               (north[:, None] - north[None, :]) ** 2) / 1000.0
np.fill_diagonal(DIST, np.inf)

vis = ~is_masked
vis_idx = np.where(vis)[0]

rows = []
for k in [1, 2, 3]:  # k-th nearest neighbour
    for label, neighbour_set in [("MR=0.0 (all)", np.arange(N)),
                                  ("MR=0.5 (visible only)", vis_idx)]:
        dists_k = []
        for si in vis_idx:
            d = DIST[si, neighbour_set]
            d_sorted = np.sort(d[np.isfinite(d)])
            if len(d_sorted) >= k:
                dists_k.append(d_sorted[k - 1])
            else:
                dists_k.append(np.nan)
        dists_k = np.array(dists_k)
        rows.append({
            "k-th NN": k,
            "Eval setup": label,
            "N stations": int(vis.sum()),
            "Mean [km]": np.nanmean(dists_k),
            "Median [km]": np.nanmedian(dists_k),
            "Std [km]": np.nanstd(dists_k),
            "Min [km]": np.nanmin(dists_k),
            "Max [km]": np.nanmax(dists_k),
            "P25 [km]": np.nanpercentile(dists_k, 25),
            "P75 [km]": np.nanpercentile(dists_k, 75),
        })

df_nn = pd.DataFrame(rows)
print(f"NN distance distribution for {int(vis.sum())} visible stations\n")
display(df_nn.style.format({
    "Mean [km]": "{:.1f}", "Median [km]": "{:.1f}", "Std [km]": "{:.1f}",
    "Min [km]": "{:.1f}", "Max [km]": "{:.1f}",
    "P25 [km]": "{:.1f}", "P75 [km]": "{:.1f}",
}).set_caption("Distance to k-th nearest neighbour — "
               "MR=0.0 uses full network, MR=0.5 uses visible subset only"))
C.save_table(df_nn, "43_nn_dist_visible_stations")

In [ ]:
# ── Bar plot: NN distance distribution by distance cluster ────────────────────
# For each k-th NN (1, 2, 3), count visible stations falling into distance
# bins under MR=0.0 vs MR=0.5.
DIST_BINS = [0, 5, 10, 15, 20, 30, 50, 100, 200]
BIN_LABELS = [f"{lo}–{hi} km" for lo, hi in zip(DIST_BINS[:-1], DIST_BINS[1:])]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ki, k in enumerate([1, 2, 3]):
    ax = axes[ki]
    x = np.arange(len(BIN_LABELS))
    w = 0.35

    for si, (label, neighbour_set, color) in enumerate([
            ("MR=0.0 (all)", np.arange(N), COL_VISIBLE),
            ("MR=0.5 (visible only)", np.where(vis)[0], COL_MASKED)]):
        dists_k = []
        for s in vis_idx:
            d = DIST[s, neighbour_set]
            d_sorted = np.sort(d[np.isfinite(d)])
            if len(d_sorted) >= k:
                dists_k.append(d_sorted[k - 1])
            else:
                dists_k.append(np.nan)
        dists_k = np.array(dists_k)
        counts, _ = np.histogram(dists_k[np.isfinite(dists_k)], bins=DIST_BINS)
        ax.bar(x + si * w, counts, width=w, color=color, alpha=0.8,
               edgecolor="white", linewidth=0.5, label=label)
        # Annotate counts
        for xi, cnt in enumerate(counts):
            if cnt > 0:
                ax.text(xi + si * w, cnt + 0.5, str(cnt),
                        ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x + w / 2)
    ax.set_xticklabels(BIN_LABELS, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"NN k={k}", fontsize=11)
    ax.set_xlabel("Distance to k-th nearest neighbour", fontsize=9)
    ax.grid(alpha=0.3, axis="y")

axes[0].set_ylabel("Number of visible stations")
axes[-1].legend(fontsize=8, loc="upper right", framealpha=0.85)
fig.suptitle("NN distance distribution for visible stations — "
             "MR=0.0 (full network) vs MR=0.5 (visible only)",
             y=1.02, fontsize=11)
plt.tight_layout()
C.save_fig(fig, "43_nn_dist_barplot"); plt.show(); plt.close(fig)


In [ ]:
# ── MAE by NN distance cluster for k=1,2,3 — v31 MR=0.0 vs v27 MR=0.5 ──────
a0 = AGG["v31"]; a5 = AGG5["v27"]
DIST_BINS = [0, 5, 10, 15, 20, 30, 50, 100, 200]
BIN_LABELS = [f"{lo}–{hi}" for lo, hi in zip(DIST_BINS[:-1], DIST_BINS[1:])]

fig, axes = plt.subplots(3, NV, figsize=(4.5 * NV, 4.0 * 3),
                         sharey="col", sharex=True)

for ki, k in enumerate([1, 2, 3]):
    # Compute k-th NN distance under MR=0.5 (visible subset)
    dists_k = np.full(N, np.nan)
    for si in vis_idx:
        d = DIST[si, vis_idx]
        d_sorted = np.sort(d[np.isfinite(d)])
        if len(d_sorted) >= k:
            dists_k[si] = d_sorted[k - 1]

    # Bin each visible station
    bin_idx = np.digitize(dists_k, DIST_BINS) - 1  # 0-based bin

    for vi, v in enumerate(VARS):
        ax = axes[ki, vi]
        x = np.arange(len(BIN_LABELS))
        w = 0.35

        for si_bar, (label, cnt_key, sum_key, color) in enumerate([
                ("v31 MR=0", "mod_all_cnt", "mod_all_sum_phys", COL_VISIBLE),
                ("v27 MR=0.5", "mod_vis_cnt", "mod_vis_sum_phys", COL_MASKED)]):
            agg = a0 if si_bar == 0 else a5
            mae_per_bin = []
            for bi in range(len(BIN_LABELS)):
                in_bin = vis & (bin_idx == bi)
                s = agg[sum_key][REF_KI, in_bin, vi].sum()
                c = agg[cnt_key][REF_KI, in_bin, vi].sum()
                mae_per_bin.append(s / c if c > 0 else np.nan)
            mae_per_bin = np.array(mae_per_bin)
            valid_bins = np.isfinite(mae_per_bin)
            ax.bar(x[valid_bins] + si_bar * w, mae_per_bin[valid_bins],
                   width=w, color=color, alpha=0.8,
                   edgecolor="white", linewidth=0.5,
                   label=label if ki == 0 and vi == NV - 1 else None)

        ax.set_xticks(x + w / 2)
        if ki == 2:
            ax.set_xticklabels(BIN_LABELS, rotation=45, ha="right", fontsize=7)
            ax.set_xlabel("NN distance [km]", fontsize=8)
        if ki == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if vi == 0:
            ax.set_ylabel(f"k={k}\nMAE at {LEAD[REF_KI]}", fontsize=9)
        ax.grid(alpha=0.3, axis="y")

axes[0, -1].legend(fontsize=8, loc="upper right", framealpha=0.85)
fig.suptitle(f"Pooled MAE at {LEAD[REF_KI]} by distance to the k-th nearest visible station — Dense (MR=0) vs MAE Transformer (MR=0.5, visible windows) "
             f"at {LEAD[REF_KI]}\n"
             f"Stations grouped by distance to k-th nearest visible neighbour",
             y=1.02, fontsize=11)
plt.tight_layout()
C.save_fig(fig, "43_nn_dist_mae_v31"); plt.show(); plt.close(fig)


In [ ]:
# ── Scatter: MAE vs k-th NN distance — v31 MR=0.0 vs v27 MR=0.5 with segments ─
# Each point uses its OWN network's k-th NN distance:
#   v31 MR=0.0 ○ → full network NN distance (all 155 stations)
#   v27 MR=0.5 △ → visible-only NN distance (~78 stations)
# Diagonal segments show joint shift: rightward = lost proximity, upward = worse MAE.
# Only visible stations (in both setups) are plotted.
a0 = AGG["v31"]; a5 = AGG5["v27"]

fig, axes = plt.subplots(3, NV, figsize=(4.5 * NV, 4.0 * 3),
                         sharex="row", sharey="col")

for ki, k in enumerate([1, 2, 3]):
    # k-th NN distance under MR=0.0 (full network)
    dists_k_full = np.full(N, np.nan)
    for si in vis_idx:
        d = DIST[si, :].copy()
        d_sorted = np.sort(d[np.isfinite(d)])
        if len(d_sorted) >= k:
            dists_k_full[si] = d_sorted[k - 1]

    # k-th NN distance under MR=0.5 (visible subset only)
    dists_k_vis = np.full(N, np.nan)
    for si in vis_idx:
        d = DIST[si, vis_idx]
        d_sorted = np.sort(d[np.isfinite(d)])
        if len(d_sorted) >= k:
            dists_k_vis[si] = d_sorted[k - 1]

    for vi, v in enumerate(VARS):
        ax = axes[ki, vi]
        cnt0 = a0["mod_all_cnt"][REF_KI, :, vi]
        mae0 = np.full(N, np.nan)
        ok0 = cnt0 > 0
        mae0[ok0] = a0["mod_all_sum_phys"][REF_KI, ok0, vi] / cnt0[ok0]
        cnt5 = a5["mod_vis_cnt"][REF_KI, :, vi]
        mae5 = np.full(N, np.nan)
        ok5 = cnt5 > 0
        mae5[ok5] = a5["mod_vis_sum_phys"][REF_KI, ok5, vi] / cnt5[ok5]
        valid = vis & ok0 & ok5 & np.isfinite(dists_k_full) & np.isfinite(dists_k_vis)
        # Diagonal segments connecting same station
        for si in np.where(valid)[0]:
            ax.plot([dists_k_full[si], dists_k_vis[si]],
                    [mae0[si], mae5[si]],
                    color="0.7", lw=0.5, zorder=3)
        ax.scatter(dists_k_full[valid], mae0[valid], s=20, alpha=0.6,
                   color=COL_VISIBLE, marker="o", zorder=4,
                   label="Dense MR=0" if ki == 0 and vi == NV - 1 else None)
        ax.scatter(dists_k_vis[valid], mae5[valid], s=20, alpha=0.6,
                   color=COL_MASKED, marker="^", zorder=5,
                   label="MAE Tr. MR=0.5" if ki == 0 and vi == NV - 1 else None)
        ax.grid(alpha=0.3)
        if ki == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if vi == 0:
            ax.set_ylabel(f"k={k}\nMAE at {LEAD[REF_KI]}", fontsize=9)
        if ki == 2:
            ax.set_xlabel("Distance to k-th NN [km]", fontsize=9)

axes[0, -1].legend(fontsize=8, loc="upper left", framealpha=0.85)
fig.suptitle(f"Per-station MAE vs k-th NN distance at {LEAD[REF_KI]} — Dense vs MAE Transformer MR=0.5\n"
             f"○ teal = v31 MR=0.0 (full network), △ coral = v27 MR=0.5 (visible only)\n"
             f"segments: rightward = lost proximity, upward = worse MAE",
             y=1.03, fontsize=11)
plt.tight_layout()
C.save_fig(fig, "43_nn_dist_mae_scatter_v31"); plt.show(); plt.close(fig)


## Masked vs visible MAE — v27 at MR=0.5

In [ ]:
# ── Masked vs visible MAE by init hour — v27 MR=0.5 ──────────────────────────
# Init bands matching 41_: 05, 11, 17, 23 UTC (±1 h).
INIT_BANDS = [(4, 6, "05 UTC"), (10, 12, "11 UTC"),
              (16, 18, "17 UTC"), (22, 24, "23 UTC")]

# Load extended aggregator (has TOD splits, but we need finer init-hour control)
# → use dump to filter by init hour
d5 = C.load_dump("v27", "mr0.50")
P = d5["preds"]; T = d5["targets"]; M = d5["masks"]
TH = d5["target_hours"]
MI = d5.get("masked_idx")
Mw, Kd, Nd, NVd = P.shape
import torch

# Build per-init-band, per-station, msk/vis MAE
ns_ = C.norm_stats(); STD_ = ns_["std"]
grid = d5["delta_steps"][0].numpy().astype(int)

fig, axes = plt.subplots(len(INIT_BANDS), NV,
                         figsize=(17, 3.0 * len(INIT_BANDS)),
                         sharex=True, sharey="col")

for bi, (lo, hi, lbl) in enumerate(INIT_BANDS):
    t0h = TH[:, 0].numpy() % 24
    sel = (t0h >= lo) & (t0h < hi)
    if sel.sum() == 0:
        continue
    Ps = P[sel]; Ts = T[sel]; Ms = M[sel]
    MIs = MI[sel] if MI is not None else None

    for vi, v in enumerate(VARS):
        ax = axes[bi, vi]
        # STD_ is (N_stations, V) per-station physical std. Two bugs here:
        # (1) STD_[vi] indexed the STATION axis instead of the variable axis
        #     (grabs one station's std across all 5 variables — shape (5,),
        #     which can't broadcast against the (Mw, K, N) error tensor: that's
        #     the ValueError). Fixed by indexing STD_[:, vi] -> shape (N,).
        # (2) Ps/Ts are torch tensors, but multiplying a torch tensor by a
        #     bare numpy array makes numpy (not torch) run the operation via
        #     the tensor's __array__ conversion, silently turning e_phys into
        #     a plain ndarray — which then breaks the `.numpy()` calls used
        #     on it further down. Wrap the std slice in torch.as_tensor(...)
        #     so the multiplication stays a torch op and e_phys stays a tensor.
        std_v = torch.as_tensor(STD_[:, vi], dtype=Ps.dtype)
        e_phys = (Ps[:, :, :, vi] - Ts[:, :, :, vi]).abs() * std_v
        mk = Ms[:, :, :, vi]  # valid mask

        # Build masked/visible masks per window
        if MIs is not None:
            msk_mask = torch.zeros(Ps.shape[0], Nd, dtype=torch.bool)
            for wi in range(Ps.shape[0]):
                msk_mask[wi, MIs[wi]] = True
            vis_mask = ~msk_mask
        else:
            vis_mask = torch.ones(Ps.shape[0], Nd, dtype=torch.bool)
            msk_mask = torch.zeros(Ps.shape[0], Nd, dtype=torch.bool)

        # Pool over windows → per-lead MAE for msk and vis
        mae_msk = np.full(Kd, np.nan)
        mae_vis = np.full(Kd, np.nan)
        for ki in range(Kd):
            e_k = e_phys[:, ki, :].numpy()
            m_k = mk[:, ki, :].numpy()
            mm = msk_mask.numpy()
            vm = vis_mask.numpy()
            s_m = (e_k * m_k * mm).sum()
            c_m = (m_k * mm).sum()
            s_v = (e_k * m_k * vm).sum()
            c_v = (m_k * vm).sum()
            if c_m > 0: mae_msk[ki] = s_m / c_m
            if c_v > 0: mae_vis[ki] = s_v / c_v

        ax.plot(range(1, Kd), mae_msk[1:], "s-", ms=3, lw=1.3,
                color=COL_MASKED, label="masked" if bi == 0 and vi == NV-1 else None)
        ax.plot(range(1, Kd), mae_vis[1:], "o-", ms=3, lw=1.3,
                color=COL_VISIBLE, label="visible" if bi == 0 and vi == NV-1 else None)
        ax.set_xticks(range(1, Kd, 2))
        if bi == len(INIT_BANDS) - 1:
            ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
        if bi == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if vi == 0:
            ax.set_ylabel(f"init {lbl}\nMAE [phys]", fontsize=9)
        ax.grid(alpha=.3)

axes[0, -1].legend(fontsize=7.5)
fig.suptitle("MAE Transformer at MR=0.5: MAE vs lead time while masked vs while visible, by forecast-origin band (±1 h around 05/11/17/23 UTC)", y=1.02)
plt.tight_layout()
C.save_fig(fig, "43_masked_vs_visible_by_init")
plt.show(); plt.close(fig)
del d5, P, T, M, TH, MI

## Masked vs visible MAE stratified by terrain class

Does masking disproportionately affect stations in particular
topographic settings? For each terrain class × variable, we report
pooled MAE for masked and visible stations, the ΔMAE (masked − visible),
and the sample size (number of stations in each group).

Groups with fewer than 5 stations are flagged — interpret with caution.

In [ ]:
# ── Terrain-class stratified masked vs visible analysis ────────────────────
TC_IDX = C.terrain_class_indices(stn)
a5 = AGG5["v27"]

# ── Summary table ────────────────────────────────────────────────────────────
# For each terrain class × variable, pool MAE over masked and visible stations.
rows = []
for tc_name, tc_idx in TC_IDX.items():
    # Split terrain-class stations into masked and visible
    tc_masked  = tc_idx[is_masked[tc_idx]]
    tc_visible = tc_idx[~is_masked[tc_idx]]
    for vi, v in enumerate(VARS):
        # Pooled MAE at reference lead for masked stations in this terrain class
        sm = a5["mod_msk_sum_phys"][REF_KI, tc_masked, vi].sum()
        cm = a5["mod_msk_cnt"][REF_KI, tc_masked, vi].sum()
        mae_m = sm / max(cm, 1) if cm > 0 else np.nan
        # Pooled MAE for visible stations in this terrain class
        sv = a5["mod_vis_sum_phys"][REF_KI, tc_visible, vi].sum()
        cv = a5["mod_vis_cnt"][REF_KI, tc_visible, vi].sum()
        mae_v = sv / max(cv, 1) if cv > 0 else np.nan
        delta = mae_m - mae_v if not (np.isnan(mae_m) or np.isnan(mae_v)) else np.nan
        flag = "†" if len(tc_masked) < 5 or len(tc_visible) < 5 else ""
        rows.append({"Terrain class": tc_name,
                     "Variable": v,
                     "N_masked": len(tc_masked),
                     "N_visible": len(tc_visible),
                     "MAE_masked": mae_m,
                     "MAE_visible": mae_v,
                     "ΔMAE": delta,
                     "flag": flag})

df_tc = pd.DataFrame(rows)
REF_LEAD = LEAD[REF_KI]
print(f"\nMasked vs visible MAE at {REF_LEAD}, stratified by terrain class (v27, MR=0.5)\n")
display(df_tc.style.format({"MAE_masked": "{:.3f}", "MAE_visible": "{:.3f}",
                            "ΔMAE": "{:+.3f}"}).set_caption(
    f"† = fewer than 5 stations in one group — interpret with caution"))
C.save_table(df_tc, "43_terrain_masked_vs_visible")

# ── Compact figure: ΔMAE by terrain class × variable ─────────────────────────
# Grouped bar plot: terrain classes on x-axis, one bar per variable,
# bar height = ΔMAE (masked − visible). Error bars via bootstrap would
# require per-window data; instead we annotate sample sizes.
tc_names = list(TC_IDX.keys())
n_tc = len(tc_names)
x = np.arange(n_tc)
w = 0.15

fig, ax = plt.subplots(figsize=(12, 4.5))
for vi, v in enumerate(VARS):
    deltas = []
    for tc_name in tc_names:
        row = df_tc[(df_tc["Terrain class"] == tc_name) &
                    (df_tc["Variable"] == v)]
        deltas.append(row["ΔMAE"].values[0])
    color = plt.cm.Set2(vi / NV)
    bars = ax.bar(x + vi * w, deltas, width=w, label=f"{v} [{C.UNITS[v]}]",
                  color=color, edgecolor="k", linewidth=0.4)

ax.axhline(0, ls=":", color="k", lw=0.8)
ax.set_xticks(x + w * (NV - 1) / 2)
ax.set_xticklabels(tc_names, fontsize=9)
ax.set_ylabel(f"ΔMAE (masked − visible) at {REF_LEAD}")
ax.set_title("Masking penalty by terrain class — MAE Transformer, MR=0.5", fontsize=11)
ax.legend(fontsize=7, ncol=NV, loc="upper left")
ax.grid(alpha=.3, axis="y")

# Annotate sample sizes below each terrain-class group
for ti, tc_name in enumerate(tc_names):
    tc_idx_arr = TC_IDX[tc_name]
    nm = is_masked[tc_idx_arr].sum()
    nv = (~is_masked[tc_idx_arr]).sum()
    ax.text(ti + w * (NV - 1) / 2, ax.get_ylim()[0],
            f"n={nm}m/{nv}v", ha="center", va="top", fontsize=7, color="0.4")

plt.tight_layout()
C.save_fig(fig, "43_terrain_masking_penalty")
plt.show()

# ── Per-variable panels: masked vs visible MAE by lead, per terrain class ──
fig, axes = plt.subplots(NV, n_tc, figsize=(4.2 * n_tc, 3.0 * NV),
                         sharex=True, squeeze=False)
for vi, v in enumerate(VARS):
    for ti, tc_name in enumerate(tc_names):
        ax = axes[vi, ti]
        tc_idx_arr = TC_IDX[tc_name]
        tc_m = tc_idx_arr[is_masked[tc_idx_arr]]
        tc_v = tc_idx_arr[~is_masked[tc_idx_arr]]
        # Pooled MAE vs lead for masked stations
        sm = a5["mod_msk_sum_phys"][:, tc_m, vi].sum(axis=1)
        cm = a5["mod_msk_cnt"][:, tc_m, vi].sum(axis=1)
        mae_m = np.where(cm > 0, sm / np.maximum(cm, 1), np.nan)
        # Pooled MAE vs lead for visible stations
        sv = a5["mod_vis_sum_phys"][:, tc_v, vi].sum(axis=1)
        cv = a5["mod_vis_cnt"][:, tc_v, vi].sum(axis=1)
        mae_v = np.where(cv > 0, sv / np.maximum(cv, 1), np.nan)
        ax.plot(range(1, K), mae_m[1:], "s-", ms=3, lw=1.2, color=COL_MASKED,
                label="masked" if ti == 0 else None)
        ax.plot(range(1, K), mae_v[1:], "o-", ms=3, lw=1.2, color=COL_VISIBLE,
                label="visible" if ti == 0 else None)
        ax.grid(alpha=.3)
        if vi == 0:
            nm, nv = len(tc_m), len(tc_v)
            flag = " †" if nm < 5 or nv < 5 else ""
            ax.set_title(f"{tc_name}\n({nm}m/{nv}v){flag}", fontsize=9)
        if ti == 0:
            ax.set_ylabel(f"{v}\n[{C.UNITS[v]}]", fontsize=9)
        if vi == NV - 1:
            ax.set_xticks(range(1, K, 2))
            ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6)

axes[0, 0].legend(fontsize=7)
fig.suptitle("Masked vs visible MAE by terrain class — MAE Transformer, MR=0.5\n"
             "† = fewer than 5 stations in group", y=1.02, fontsize=11)
plt.tight_layout()
C.save_fig(fig, "43_terrain_masked_vs_visible_by_lead")
plt.show()


## A — Missing-station penalty

v27 MR=0.5 masked vs v27 MR=0 (all stations visible).

*How much does removing the target station's observations hurt
performance?* Both curves use the same model weights — the only
difference is whether the station's own observations are available.

In [ ]:
fig, axes = plt.subplots(1, NV, figsize=(17, 3.4))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    m27_msk = C.metric(AGG5["v27"], "mae", sub="msk", pool=("N",))[:, vi]
    m27_mr0 = C.metric(AGG["v27"], "mae", pool=("N",))[:, vi]
    ax.plot(range(1, K), m27_msk[1:], "s-", ms=3, lw=1.3, color=COL_MASKED,
            label="MAE Tr. MR=0.5 (masked stations)")
    ax.plot(range(1, K), m27_mr0[1:], "o-", ms=3, lw=1.3, color=COL_VISIBLE,
            label="MAE Tr. MR=0 (all visible)")
    ax.set_xticks(range(1, K, 2))
    ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10); ax.grid(alpha=.3)
axes[0].set_ylabel("MAE [phys]"); axes[-1].legend(fontsize=6.5)
fig.suptitle("Missing-station penalty — same weights, different information", y=1.04)
plt.tight_layout(); C.save_fig(fig, "43_missing_station_penalty"); plt.show()

## B — Operational comparison: v27 MR=0.5 masked vs v31 MR=0

**Not a controlled training ablation** — the evaluation conditions differ
(v27 has 50 % of stations removed; v31 sees all). This shows what happens
operationally when a masking-trained model faces actual station outages
compared to a dense-trained model in ideal conditions.

In [ ]:
fig, axes = plt.subplots(1, NV, figsize=(17, 3.4))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    m27_msk = C.metric(AGG5["v27"], "mae", sub="msk", pool=("N",))[:, vi]
    m31_all = C.metric(AGG["v31"], "mae", pool=("N",))[:, vi]
    m27_all = C.metric(AGG["v27"], "mae", pool=("N",))[:, vi]
    ax.plot(range(1, K), m27_msk[1:], "s-", ms=3, lw=1.3, color=COL_MASKED,
            label="MAE Tr. masked (MR=0.5)")
    ax.plot(range(1, K), m27_all[1:], "o-", ms=3, lw=1.3, color="#1F5F6B",
            label="MAE Tr. all (MR=0)")
    ax.plot(range(1, K), m31_all[1:], "D-", ms=3, lw=1.3, color="#D9663D",
            label="Dense (MR=0)")
    ax.set_xticks(range(1, K, 2))
    ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10); ax.grid(alpha=.3)
axes[0].set_ylabel("MAE [phys]"); axes[-1].legend(fontsize=6.5)
fig.suptitle("Operational comparison (different eval conditions — not a controlled ablation)",
             y=1.04)
plt.tight_layout(); C.save_fig(fig, "43_operational_comparison"); plt.show()

## Interpretation

**Masking penalty** is largest at Δ=0 (reconstruction: masked stations
have no persistence shortcut) and shrinks with lead time as temporal
extrapolation dominates.

**Missing-station penalty** (plot A) isolates the information cost: same
model weights, only whether the station's own observations are available
changes.

**Operational comparison** (plot B) mixes two effects — training regime
and evaluation information — so should not be interpreted as a
controlled ablation. v31 has no MR=0.5 evaluation, preventing a
controlled masking-training comparison.

## Stations excluded from evaluation, by variable

Marks every station×variable pair dropped by `DROP_SV` (see common.py) with
an X on the map: **black** = missing in both train and test (>50%),
**white** = missing in train only (sensor added after 2021, present in
test), **grey** = missing in test only (no such pair exists in the current
data — kept for completeness), **orange** = present in both splits but
excluded for a data-quality reason (BIZ/pressure, systematic sensor drift),
not for missingness.

In [ ]:
# ── Stations excluded from evaluation, by variable ──────────────────────────
REASON_STYLE = {
    "missing_train": dict(color="white",   label="missing in train only"),
    "missing_test":  dict(color="0.6",     label="missing in test only"),
    "missing_both":  dict(color="black",   label="missing in train & test"),
    "drift":         dict(color="#E8A838", label="excluded \u2014 sensor drift"),
}
COL_INCLUDED = "#2E86C1"  # blue for included stations
EXCL_REASONS = C.excluded_station_variable_reasons()

# Switzerland's LV95 extent is landscape (~350 km E-W x 220 km N-S, aspect
# ~1.6:1), but figsize height 7.5 gave each panel a portrait box — with
# draw_dem's equal-aspect plotting, that left the map letterboxed with a
# huge blank margin above/below it, so the legend anchored to the bottom
# of the panel ended up far below the visible map. Shrinking the height
# to match the map's own aspect removes that dead space.
fig, axes = plt.subplots(1, NV, figsize=(6.0 * NV, 4.6))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    draw_dem(ax, dem_bg, dem_fg, switzerland)

    # Find excluded stations for this variable
    excl_abbrs = {abbr for (abbr, var), reason in EXCL_REASONS.items() if var == v}
    is_excl = np.array([a in excl_abbrs for a in stn.abbr])

    # All included stations: circles
    incl = ~is_excl
    ax.scatter(stn.easting[incl], stn.northing[incl],
               c=COL_INCLUDED, marker="o", s=60,
               edgecolors="white", linewidths=0.5, zorder=5,
               label="included" if vi == 0 else None)

    # Excluded stations: X markers, coloured by reason
    for abbr_idx in np.where(is_excl)[0]:
        abbr = stn.abbr.iloc[abbr_idx]
        reason = EXCL_REASONS.get((abbr, v), "missing_both")
        style = REASON_STYLE[reason]
        ax.scatter(stn.easting.iloc[abbr_idx], stn.northing.iloc[abbr_idx],
                   c=style["color"], marker="X", s=100,
                   edgecolors="0.3", linewidths=0.6, zorder=6,
                   label=style["label"] if vi == 0 else None)

    ax.set_title(f"{v}", fontsize=12)

# De-duplicate legend entries
from collections import OrderedDict
handles, labels = axes[0].get_legend_handles_labels()
by_label = OrderedDict(zip(labels, handles))
fig.legend(by_label.values(), by_label.keys(), loc="lower center",
           ncol=len(by_label), fontsize=9, bbox_to_anchor=(0.5, -0.06), frameon=True)
#fig.suptitle("Station inclusion/exclusion by variable \u2014 all stations", y=1.02)
plt.tight_layout(rect=[0, 0.06, 1, 1])
C.save_fig(fig, "excluded_stations_map")
plt.show()

print("Excluded station\u00d7variable pairs:")
for (abbr, v), reason in sorted(EXCL_REASONS.items()):
    print(f"  {abbr:<4} {v:<12} {reason}")